# Interaktywna Wizualizacja Uprawnień Looker (Sankey)

Poniższy kod wczytuje plik `looker_graph_all.json` wygenerowany przez skrypt w Pythonie i tworzy interaktywny diagram Sankeya z użyciem biblioteki `Plotly`.


In [ ]:
import json
import plotly.graph_objects as go

# 1. Wczytanie danych wygenerowanych przez skrypt Pythona
file_name = "looker_graph_all.json" 
try:
    with open(file_name, "r") as f:
        graph_data = json.load(f)
except FileNotFoundError:
    raise FileNotFoundError(f"Plik {file_name} nie istnieje! Uruchom najpierw: python3 permissions_graph_extractor.py")

# Kolory
type_colors = {
    "model": "#ff4d4d",
    "explore": "#ffa64d",
    "dashboard": "#4d4dff",
    "group": "#ff4dff",
    "user": "#8B4513",
    "folder": "#4dff4d",
    "model_set": "#8B00FF",
    "role": "#00FFFF",
    "user_attribute": "#FFC0CB",
    "access_grant": "#FFFF00"
}

nodes_data = graph_data.get("nodes", [])
edges_data = graph_data.get("edges", [])

# 2. Mapowanie ID (string) na Indeksy (integer)
node_mapping = {}
node_labels = []
node_colors = []

for idx, node in enumerate(nodes_data):
    node_mapping[node["id"]] = idx
    label = node.get("label", node["id"])
    members = node.get("members", [])
    if members:
        label += f" ({len(members)} users)"
        
    node_labels.append(label)
    node_colors.append(type_colors.get(node.get("type"), "#cccccc"))

# 3. Budowanie krawędzi
sources = []
targets = []
values = []

for edge in edges_data:
    if edge["source"] in node_mapping and edge["target"] in node_mapping:
        sources.append(node_mapping[edge["source"]])
        targets.append(node_mapping[edge["target"]])
        values.append(1)

# 4. Wygenerowanie diagramu Sankey
fig = go.Figure(data=[go.Sankey(
    node = dict(
      pad = 20,
      thickness = 30,
      line = dict(color = "black", width = 0.5),
      label = node_labels,
      color = node_colors
    ),
    link = dict(
      source = sources,
      target = targets,
      value = values,
      color = "rgba(200, 200, 200, 0.4)"
    )
)])

fig.update_layout(
    title_text="Przepływ Uprawnień Looker", 
    font_size=12,
    height=900
)

fig.show()
fig.write_html("interactive_sankey_graph.html")
